# Чат-бот на базе rugpt3small_based_on_gpt2

## Среда

### Установка бибилиотек

In [1]:
# pip install transformers datasets torch rouge detoxify nltk bert-score

### Стандартные бибилотеки

In [2]:
import time
import re
import json
import threading
import sys
import itertools

### Обработка данных

In [3]:
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

### Нейросеть

In [4]:
import torch
from torch.cuda.amp import GradScaler
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel, 
    GPT2Config,
    Trainer, 
    TrainingArguments, 
    DataCollatorForLanguageModeling
)

### Визуализация

In [5]:
from tqdm.notebook import tqdm

### Метрики

In [6]:
from rouge import Rouge
from rouge_score import rouge_scorer
from detoxify import Detoxify
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from bert_score import score

## Замеры времени

In [7]:
# Фрейм для замеров времени
time_metrics = pd.DataFrame(columns=["stage", "time_sec", "time_formatted"])

In [8]:
# Функция для форматирования времени
def format_time(seconds):
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    milliseconds = int((seconds - int(seconds)) * 100)
    return f"{int(hours):02}:{int(minutes):02}:{int(seconds):02}.{milliseconds:02}"

In [9]:
# Функция для добавления времени
def add_time_metric(stage_name, time_value):
    formatted_time = format_time(time_value)
    new_row = pd.DataFrame({
        "stage": [stage_name],
        "time_sec": [round(time_value, 2)],
        "time_formatted": [formatted_time]
    })
    global time_metrics
    time_metrics = pd.concat([time_metrics, new_row], ignore_index=True)

In [10]:
start_total_time = time.time()

## Данные

### Загрузка датасета

In [11]:
dataset = load_dataset("MLNavigator/russian-retrieval")

### Очистка датасета

In [12]:
# Очистка датасета от меток SOURCE
def clean_dataset(example):
    example['q'] = re.sub(r'\s*SOURCE.*\n*.*', '', example['q'], flags=re.IGNORECASE).strip()
    example['a'] = re.sub(r'\s*SOURCE.*\n*.*', '', example['a'], flags=re.IGNORECASE).strip()
    return example

dataset = dataset.map(clean_dataset)

### Уменьшение датасета для ускорения тестирования

In [13]:
dataset['train'] = dataset['train'].select(range(10000))

### Предобработка

In [14]:
# Разделение на train/val/test (80%/10%/10%)
split_dataset = dataset['train'].train_test_split(test_size=0.2, seed=42)
train_val_split = split_dataset['test'].train_test_split(test_size=0.5, seed=42)

train_data = split_dataset['train']
val_data = train_val_split['train']
test_data = train_val_split['test']

# Подготовка данных
tokenizer = GPT2Tokenizer.from_pretrained("ai-forever/rugpt3small_based_on_gpt2")
tokenizer.pad_token = tokenizer.eos_token  # Установка pad_token

def preprocess_function(examples):
    inputs = [
        f"Контекст: {context}\nВопрос: {q}\nОтвет: {a}" 
        for context, q, a in zip(examples['context'], examples['q'], examples['a'])
    ]
    tokenized = tokenizer(
        inputs,
        truncation=True,
        padding="max_length",
        max_length=256,
        return_attention_mask=True
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

# Предобработка на данных
train_data = train_data.map(preprocess_function, batched=True)
val_data = val_data.map(preprocess_function, batched=True)
test_data = test_data.map(preprocess_function, batched=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

## Модель

### Загрузка пердобученной модели

In [15]:
model = GPT2LMHeadModel.from_pretrained("ai-forever/rugpt3small_based_on_gpt2")

### Оптимизация для GPU или CPU

In [16]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
device

'cuda'

In [17]:
# Очистка кэша PyTorch
torch.cuda.empty_cache()

### Настройка параметров обучения с учётом параметров RTX2060

In [18]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=400,
    save_total_limit=2,
    fp16=True if device == "cuda" else False,
    gradient_accumulation_steps=4,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to= "none",
)

### Обучение модели

In [19]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # GPT-2 не использует masked language modeling
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=val_data,
    data_collator=data_collator,  # data_collator вместо tokenizer
)

In [20]:
start_train_time = time.time()

In [21]:
trainer.train()
trainer.save_model("results/final_model")

Step,Training Loss,Validation Loss
200,3.271800,3.103315
400,3.124700,3.038504
600,2.955700,3.010733
800,2.869300,2.981751
1000,2.933600,2.942219
1200,2.753400,2.938383
1400,2.747300,2.918231
1600,2.568500,2.925013
1800,2.622300,2.903335
2000,2.609600,2.896843


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


In [22]:
end_train_time = time.time()
train_duration = end_train_time - start_train_time
print(f"Обучение заняло: {train_duration:.2f} сек")
add_time_metric("Обучение", train_duration)

Обучение заняло: 10119.80 сек


In [23]:
# Сохранения логов обучения
training_logs = trainer.state.log_history

# Сохранение в DataFrame и CSV
df_logs = pd.DataFrame([log for log in training_logs if 'loss' in log or 'eval_loss' in log])
df_logs.to_csv("rugpt_training_logs.csv", index=False)

print("Результаты обучения сохранены в rugpt_training_logs.csv")
df_logs

Результаты обучения сохранены в rugpt_training_logs.csv


,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second
0,3.4784,8.387797,5.000000e-06,0.02,10,NaN,NaN,NaN,NaN
1,3.3937,7.434908,1.000000e-05,0.04,20,NaN,NaN,NaN,NaN
2,3.3799,7.540124,1.500000e-05,0.06,30,NaN,NaN,NaN,NaN
3,3.3524,7.285983,2.000000e-05,0.08,40,NaN,NaN,NaN,NaN
4,3.3502,7.067327,2.500000e-05,0.10,50,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
257,2.5205,7.174774,8.333333e-07,4.92,2460,NaN,NaN,NaN,NaN
258,2.5622,6.842062,6.250000e-07,4.94,2470,NaN,NaN,NaN,NaN
259,2.5346,7.127612,4.166667e-07,4.96,2480,NaN,NaN,NaN,NaN
260,2.5667,6.867731,2.083333e-07,4.98,2490,NaN,NaN,NaN,NaN


### Генерация ответа

In [24]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:32'

In [25]:
# Функиця генерации ответа
def generate_answer(question, context=None):
    try:
        model.eval()
        input_text = f"Контекст: {context}\nВопрос: {question}\nОтвет:" if context else f"Вопрос: {question}\nОтвет:"
        inputs = tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            max_length=256,
            padding="max_length",
            return_attention_mask=True
        ).to(device)

        start_time = time.time()  # Исправлено: добавлена инициализация start_time
        
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_new_tokens=50,
                num_beams=5,
                no_repeat_ngram_size=2,
                pad_token_id=tokenizer.eos_token_id
            )
        
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
                # Расширенная очистка ответов
        clean_response = (
            response
            .replace("&ndash;", "-")      # Замена тире
            .replace("&mdash;", "—")      # Длинное тире
            .replace("&nbsp;", " ")       # Неразрывный пробел
            .replace("&laquo;", "«")      # Левые кавычки
            .replace("&raquo;", "»")      # Правые кавычки
        )
        
        # Удаление всех оставшихся HTML-сущностей
        clean_response = re.sub(r'&[\w#-]+;', '', clean_response)
        
        # Финальная очистка
        clean_response = (
            clean_response
            .replace("\\n", " ")          # Удаление переносов
            .replace("  ", " ")           # Двойные пробелы
            .strip()                      # Удаление пробелов по краям
        )

        # Обрезка после первой точки
        clean_response = re.split(r'[.!?]', clean_response.split('Ответ:')[-1], maxsplit=1)[0].strip()
        if not clean_response.endswith(('.', '!', '?')):
            clean_response += '.' 
        
        generated_answer = clean_response
        latency = time.time() - start_time  # Используется start_time
        
        return generated_answer, latency
            
    except Exception as e:
        print(f"Ошибка генерации: {e}")
        return "Не удалось сгенерировать ответ", 0.0

# Обновление функции генерации ответов для тестового датасета
def generate_answers_and_save(test_data, output_file="rugpt_generated_answers.json"):
    generated_data = []
    
    for example in tqdm(test_data, desc="Генерация ответов", unit="example", total=len(test_data)):
        q = example['q']
        context = example.get('context', '')  # Получаем контекст из датасета
        true_answer = example['a']
        
        answer, latency = generate_answer(q, context)
        
        generated_data.append({
            "Вопрос": q,
            "Контекст": context,
            "Эталонный ответ": true_answer,
            "Сгенерированный ответ": answer,
            "Время отклика (сек)": round(latency, 4)
        })
    
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(generated_data, f, ensure_ascii=False, indent=2)
    print(f"Сгенерированные ответы сохранены в {output_file}")

In [26]:
start_gen_time = time.time()

In [27]:
# Запуск генерации ответов
generate_answers_and_save(test_data)

Генерация ответов:   0%|          | 0/1000 [00:00<?, ?example/s]

Сгенерированные ответы сохранены в rugpt_generated_answers.json


In [28]:
end_gen_time = time.time()
gen_duration = end_gen_time - start_gen_time
print(f"Генерация ответов заняла: {gen_duration:.2f} сек")
add_time_metric("Генерация ответов", gen_duration)

Генерация ответов заняла: 3469.71 сек


## Оценка качества модели

### Загрузка моделей

In [29]:
# Загрузка модели Detoxify для оценки токсичности ответов
detox = Detoxify('original', device=device)
def check_toxicity(text):
    return detox.predict(text)['toxicity']

### Функции для расчета метрик

In [30]:
# Перплексия для оценки предсказания текста
def calculate_perplexity(question, generated_answer):
    inputs = tokenizer(f"Вопрос: {question} Ответ: {generated_answer}", return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
    return torch.exp(outputs.loss).item()

In [31]:
def calculate_bertscore(generated, true_answer):
    P, R, F1 = score(
        [generated], 
        [true_answer], 
        lang="ru", 
        model_type="bert-base-multilingual-cased",
        verbose=False
    )
    return F1.item()

In [32]:
# Токсичность ответов
def check_toxicity(text):
    return detox.predict(text)['toxicity']

### Расчет метрик из сохраненных данных

In [33]:
# Функция расчёта метрик 
def calculate_metrics_from_file(input_file="rugpt_generated_answers.json", output_file="rugpt_metrics.csv"):
    with open(input_file, "r", encoding="utf-8") as f:
        generated_data = json.load(f)
    
    metrics = []
    
    for example in tqdm(generated_data, desc="Расчет метрик", unit="example", total=len(generated_data)):
        q = example["Вопрос"]
        true_answer = example["Эталонный ответ"]
        generated = example["Сгенерированный ответ"]
        latency = example.get("Время отклика (сек)", 0.0)
        
        # Проверка, что на входе строка
        if not isinstance(generated, str):
            generated = " ".join(generated)
        
        # Расчет метрик
        ppl = calculate_perplexity(q, generated) 
        bert_score = calculate_bertscore(generated, true_answer)
        toxicity = check_toxicity(generated)
        
        metrics.append({
            "Вопрос": q,
            "Сгенерированный ответ": generated,
            "Эталонный ответ": true_answer,
            "Perplexity": round(ppl, 2),
            "BERTScore F1": round(bert_score, 4),
            "Токсичность": round(toxicity, 4),
            "Время отклика (сек)": round(latency, 4)
        })
    
    # Создание DataFrame и сохранение
    df = pd.DataFrame(metrics)
    df.to_csv(output_file, index=False)
    print(f"Метрики сохранены в {output_file}")

    avg_latency = df["Время отклика (сек)"].mean()

    # Сводка по средним значениям
    summary = df.mean(numeric_only=True)
    print("Средние значения метрик:")
    print(summary)

In [34]:
start_metrics_time = time.time()

### Расчёт метрик

In [35]:
calculate_metrics_from_file()

Расчет метрик:   0%|          | 0/1000 [00:00<?, ?example/s]

Метрики сохранены в rugpt_metrics.csv
Средние значения метрик:
Perplexity             13.787020
BERTScore F1            0.723513
Токсичность             0.002110
Время отклика (сек)     3.458716
dtype: float64


In [36]:
end_metrics_time = time.time()
metrics_duration = end_metrics_time - start_metrics_time
print(f"Расчет метрик занял: {metrics_duration:.2f} сек")
add_time_metric("Расчёт метрик", metrics_duration)

Расчет метрик занял: 2160.25 сек


In [37]:
end_total_time = time.time()
total_duration = end_total_time - start_total_time
add_time_metric("Общее", total_duration)

In [38]:
time_metrics

,stage,time_sec,time_formatted
0,Обучение,10119.80,02:48:39.79
1,Генерация ответов,3469.71,00:57:49.70
2,Расчёт метрик,2160.25,00:36:00.24
3,Общее,15906.86,04:25:06.85


## Чат-бот

In [39]:
def chat_bot():
    print("Бот: Здравствуйте! Для выхода введите 'выход'.")
    qa_history = []
    
    def loading_animation(stop_event):
        symbols = itertools.cycle(['⠇', '⠋', '⠙', '⠸', '⠴', '⠦'])
        while not stop_event.is_set():
            sys.stdout.write(f"\rБот: Думаю... {next(symbols)}")
            sys.stdout.flush()
            time.sleep(0.1)
        sys.stdout.write("\r" + " " * 40 + "\r")
    
    while True:
        user_input = input("\nВы: ").strip()
        if user_input.lower() == "выход":
            break
        
        # Разделение ввода на контекст и вопрос (если пользователь предоставляет контекст)
        parts = user_input.split('|')
        if len(parts) == 2:
            context, question = parts[0].strip(), parts[1].strip()
        else:
            context, question = None, user_input
        
        current_question = question
        attempts = []
        max_attempts = 3
        success = False
        clarification_used = False
        
        for attempt_num in range(1, max_attempts + 1):
            stop_animation = threading.Event()
            animation_thread = threading.Thread(target=loading_animation, args=(stop_animation,))
            animation_thread.start()
             
            try:
                answer, latency = generate_answer(current_question, context)
                clean_answer = answer.split("Ответ:")[-1].strip()
                
                stop_animation.set()
                animation_thread.join()
                
                attempts.append({
                    "attempt": attempt_num,
                    "question": current_question,
                    "answer": clean_answer,
                    "latency": round(latency, 4),
                    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
                })
                
                print(f"Бот: {clean_answer} ({latency:.2f}с)")
                
                feedback = ""
                while feedback not in ["да", "нет"]:
                    feedback = input("ℹ️ Вас устраивает ответ? (да/нет): ").lower()
                
                if feedback == "да":
                    success = True
                    clarification_used = attempt_num > 1
                    break
                else:
                    if attempt_num < max_attempts:
                        clarification = input("🔄 Уточните вопрос: ").strip()
                        current_question += f" ({clarification})"
        
            except Exception as e:
                stop_animation.set()
                animation_thread.join()
                print(f"\rОшибка генерации: {e}")
                answer = "Не удалось сгенерировать ответ"
                break
        
        record = {
            "original_question": user_input,
            "context": context,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "attempts": attempts,
            "status": "clarifications" if clarification_used else "success" if success else "operator_required"
        }
        
        if success:
            print("✅ Ответ сохранен в истории успешных" if not clarification_used else 
                  f"⚠️ Ответ с уточнениями сохранен в истории (попыток: {attempt_num})")
        else:
            print("❌ Бот: Передаю запрос оператору")
        
        qa_history.append(record)
        
        with open("rugpt_qa_full_history.json", "w", encoding="utf-8") as f:
            json.dump(qa_history, f, ensure_ascii=False, indent=2)

    print("Бот: До свидания!")

In [40]:
# Запуск чат-бота
if __name__ == "__main__":
    chat_bot()

Бот: Здравствуйте! Для выхода введите 'выход'.



Вы:  кто президент РФ с 2012 года?


Бот: Президент РФ Путин Владимир Владимирович с 2011 года является президентом Российской Федерации. (3.70с)


ℹ️ Вас устраивает ответ? (да/нет):  да


✅ Ответ сохранен в истории успешных


...

Бот: Великая Отечественная война началась в 1914 году, когда германские войска вторглись на территорию Российской империи. (3.64с)


ℹ️ Вас устраивает ответ? (да/нет):  нет


❌ Бот: Передаю запрос оператору



Вы:  выход


Бот: До свидания!
